In [ ]:


from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import random
import pickle
import zipfile
from copy import deepcopy
from typing import List

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import confusion_matrix
import torch.nn.functional as F

SEED = 42
NUM_WORKERS = 0
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4
GN_GROUPS = 32

SAVE_DIR = "/content/drive/MyDrive/ML_Project/project_files/QDA_Tiny_Imagenet"
os.makedirs(SAVE_DIR, exist_ok=True)

ZIP_PATH = "/content/drive/MyDrive/ML_Project/project_files/First_benchmark/tiny-imagenet-processed.zip"
DATA_ROOT = "/content/drive/MyDrive/ML_Project/data/TINYIMG"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True)

def torch_load_compat(path, map_location="cpu"):
    return torch.load(path, map_location=map_location, weights_only=False)

def get_best_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_best_device()
PIN_MEM = device.type == "cuda"
print(f"[INFO] Using device: {device}")

MD_DIR = SAVE_DIR
os.makedirs(MD_DIR, exist_ok=True)

# Task stats
MD_TASK1_STATS_PATH = os.path.join(MD_DIR, "best_tinyimg_qda_20class_stats.pth")
MD_TASK2_STATS_PATH = os.path.join(MD_DIR, "task2_tinyimg_best_epoch_qda_stats_classes20_39.pth")
MD_TASK3_STATS_PATH = os.path.join(MD_DIR, "task3_tinyimg_best_epoch_qda_stats_classes40_59.pth")
MD_TASK4_STATS_PATH = os.path.join(MD_DIR, "task4_tinyimg_best_epoch_qda_stats_classes60_79.pth")
MD_TASK5_STATS_PATH = os.path.join(MD_DIR, "task5_tinyimg_best_epoch_qda_stats_classes80_99.pth")
MD_TASK6_STATS_PATH = os.path.join(MD_DIR, "task6_tinyimg_best_epoch_qda_stats_classes100_119.pth")
MD_TASK7_STATS_PATH = os.path.join(MD_DIR, "task7_tinyimg_best_epoch_qda_stats_classes120_139.pth")
MD_TASK8_STATS_PATH = os.path.join(MD_DIR, "task8_tinyimg_best_epoch_qda_stats_classes140_159.pth")

# Task8 weights -> Task9 expansion source
MD_TASK8_WEIGHTS_PATH = os.path.join(MD_DIR, "finetuned_tinyimg_task8_best_qda_classes140_159.pth")

# Task9 outputs
MD_TASK9_WEIGHTS_PATH = os.path.join(MD_DIR, "finetuned_tinyimg_task9_best_qda_classes160_179.pth")
TASK9_BEST_STATS_PATH = os.path.join(MD_DIR, "task9_tinyimg_best_epoch_qda_stats_classes160_179.pth")

TOPK_PATH = os.path.join(MD_DIR, "(3K)Fisher_qda_tinyimg_classes160_topk.pkl")
NEIGHBORS_PATH = os.path.join(MD_DIR, "(3K)Fisher_qda_neighbors_tinyimg_classes160.pkl")

def make_gn(C: int) -> nn.GroupNorm:
    g = min(GN_GROUPS, C)
    while g > 1 and C % g != 0:
        g //= 2
    return nn.GroupNorm(num_groups=max(1, g), num_channels=C)

def conv3x3(in_planes, out_planes, stride=1):
    return nn.Conv2d(
        in_planes,
        out_planes,
        kernel_size=3,
        stride=stride,
        padding=1,
        bias=False
    )

class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super().__init__()

        self.conv1 = conv3x3(in_planes, planes, stride)
        self.gn1 = make_gn(planes)

        self.conv2 = conv3x3(planes, planes)
        self.gn2 = make_gn(planes)

        self.shortcut = nn.Sequential()

        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_planes,
                    planes,
                    kernel_size=1,
                    stride=stride,
                    bias=False
                ),
                make_gn(planes)
            )

    def forward(self, x):
        out = torch.relu(self.gn1(self.conv1(x)))
        out = self.gn2(self.conv2(out))
        out += self.shortcut(x)
        return torch.relu(out)

class ResNet18Backbone(nn.Module):
    def __init__(self, nf=64):
        super().__init__()

        self.nf = nf
        self.conv1 = conv3x3(3, nf)
        self.gn1 = make_gn(nf)

        self.layer1 = nn.Sequential(
            BasicBlock(nf, nf),
            BasicBlock(nf, nf)
        )
        self.layer2 = nn.Sequential(
            BasicBlock(nf, nf * 2, 2),
            BasicBlock(nf * 2, nf * 2)
        )
        self.layer3 = nn.Sequential(
            BasicBlock(nf * 2, nf * 4, 2),
            BasicBlock(nf * 4, nf * 4)
        )
        self.layer4 = nn.Sequential(
            BasicBlock(nf * 4, nf * 8, 2),
            BasicBlock(nf * 8, nf * 8)
        )

    def forward(self, x):
        x = torch.relu(self.gn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = F.avg_pool2d(x, x.shape[2])
        return x.view(x.size(0), -1)

    @property
    def out_dim(self):
        return self.nf * 8

class SingleHeadNet(nn.Module):
    def __init__(self, backbone, num_classes):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Linear(backbone.out_dim, num_classes)

    def forward(self, x):
        return self.head(self.backbone(x))

def ensure_extracted(zip_path: str, data_root: str) -> str:
    processed_dir = os.path.join(data_root, "processed")

    if os.path.isdir(processed_dir) and len(os.listdir(processed_dir)) > 0:
        print(f"[INFO] Found processed data at: {processed_dir}")
        return data_root

    if not os.path.isfile(zip_path):
        raise FileNotFoundError(f"ZIP not found at:\n{zip_path}")

    os.makedirs(data_root, exist_ok=True)
    print(f"[INFO] Extracting ZIP from:\n{zip_path}\n-> to:\n{data_root}")

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(data_root)

    processed_dir = os.path.join(data_root, "processed")
    assert os.path.isdir(processed_dir), "processed/ folder missing after unzip!"

    return data_root

DATA_ROOT = ensure_extracted(ZIP_PATH, DATA_ROOT)

TIN_IMAGENET_MEAN = (0.4802, 0.4480, 0.3975)
TIN_IMAGENET_STD = (0.2770, 0.2691, 0.2821)

class TinyImagenet(Dataset):
    def __init__(self, root: str, train: bool = True, transform=None):
        self.root = root
        self.train = train
        self.transform = transform

        split = "train" if self.train else "val"

        xs = []
        ys = []

        for num in range(20):
            xs.append(
                np.load(
                    os.path.join(
                        root,
                        f"processed/x_{split}_{num+1:02d}.npy"
                    )
                )
            )
            ys.append(
                np.load(
                    os.path.join(
                        root,
                        f"processed/y_{split}_{num+1:02d}.npy"
                    )
                )
            )

        self.data = np.concatenate(np.array(xs))
        self.targets = np.concatenate(np.array(ys)).astype(int)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        img = self.data[index]
        target = int(self.targets[index])

        img = Image.fromarray(np.uint8(255 * img))

        if self.transform is not None:
            img = self.transform(img)

        return img, target

def get_tinyimg_datasets():
    tf_train = transforms.Compose([
        transforms.RandomCrop(64, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(TIN_IMAGENET_MEAN, TIN_IMAGENET_STD)
    ])

    tf_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(TIN_IMAGENET_MEAN, TIN_IMAGENET_STD)
    ])

    train = TinyImagenet(DATA_ROOT, True, tf_train)
    test = TinyImagenet(DATA_ROOT, False, tf_test)

    return train, test

def indices_for_classes(dataset, keep):
    keep = set(keep)
    return [
        i
        for i, y in enumerate(dataset.targets)
        if int(y) in keep
    ]

class KeepOriginalLabelsDataset(Dataset):
    def __init__(self, dataset, indices):
        self.dataset = dataset
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        x, y = self.dataset[self.indices[idx]]
        return x, y

def make_loader(ds, bs, shuffle):
    return DataLoader(
        ds,
        batch_size=bs,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEM
    )

def load_fisher_data():
    with open(TOPK_PATH, "rb") as f:
        topk = pickle.load(f)

    with open(NEIGHBORS_PATH, "rb") as f:
        neigh = pickle.load(f)

    print(f"[INFO] Loaded Fisher data | TopK={len(topk)} | Neigh={len(neigh)}")
    return topk, neigh

def build_neighbor_tensors(model, fisher_neighbors, neighbor_original_values):
    pm = dict(model.named_parameters())

    usable = [
        n
        for n in fisher_neighbors
        if n["name"] in pm
    ]

    if not usable:
        return {}

    max_f = max((n["fisher"] for n in usable), default=1.0) or 1.0
    buckets = {}

    for n in usable:
        key = (n["name"], int(n["index"]))

        if key in neighbor_original_values:
            buckets.setdefault(n["name"], []).append(
                (
                    int(n["index"]),
                    float(n["fisher"]) / max_f
                )
            )

    ewc = {}

    for name, lst in buckets.items():
        lst.sort(key=lambda t: t[0])

        idxs = torch.tensor(
            [i for i, _ in lst],
            device=device,
            dtype=torch.long
        )

        fish = torch.tensor(
            [f for _, f in lst],
            device=device,
            dtype=torch.float32
        )

        flat = pm[name].view(-1)

        orig = torch.stack([
            neighbor_original_values[(name, int(i))]
            for i in idxs.tolist()
        ]).to(flat.device, dtype=flat.dtype)

        ewc[name] = {
            "idxs": idxs,
            "fish": fish,
            "orig": orig
        }

    return ewc

def build_freeze_masks_and_cache(model, topk_list):
    masks = {}
    frozen_idxs = {}
    frozen_vals = {}

    pm = dict(model.named_parameters())
    by_name = {}

    for e in topk_list:
        n = e["name"]
        i = int(e["index"])

        if n in pm and not n.startswith("head"):
            by_name.setdefault(n, []).append(i)

    for name, idxs in by_name.items():
        p = pm[name]
        flat = p.detach().view(-1)

        idxs_t = torch.tensor(
            idxs,
            device=flat.device,
            dtype=torch.long
        )

        m = torch.ones_like(
            p,
            dtype=torch.bool,
            device=p.device
        )

        mv = m.view(-1)
        mv[idxs_t] = False

        masks[name] = mv.view_as(p)
        frozen_idxs[name] = idxs_t
        frozen_vals[name] = flat.index_select(0, idxs_t).clone()

    return masks, frozen_idxs, frozen_vals

def apply_freeze_after_backward(model, masks):
    with torch.no_grad():
        for n, p in model.named_parameters():
            m = masks.get(n, None)

            if p.grad is not None and m is not None:
                p.grad.mul_(m.to(p.grad.dtype))

@torch.no_grad()
def apply_strict_freeze_after_step(param_map, frozen_idxs, frozen_vals):
    for n, idxs in frozen_idxs.items():
        if n in param_map:
            flat = param_map[n].view(-1)
            flat.index_copy_(
                0,
                idxs,
                frozen_vals[n].to(flat.device, dtype=flat.dtype)
            )

@torch.no_grad()
def compute_qda_stats_from_loader(
    backbone: nn.Module,
    loader,
    class_ids: List[int]
):
    backbone.eval()

    feats = []
    labels = []

    for x, y in loader:
        x = x.to(device)
        f = backbone(x).detach().cpu()

        feats.append(f)
        labels.append(y.detach().cpu())

    feats = torch.cat(feats, dim=0)
    labels = torch.cat(labels, dim=0)

    means = []
    inv_covs = []
    logdets = []
    priors = []

    N = feats.size(0)

    for c in class_ids:
        cf = feats[labels == c]
        Nc = cf.size(0)

        if Nc == 0:
            raise RuntimeError(
                f"No samples found for class {c} to compute QDA stats."
            )

        mean = cf.mean(0)
        centered = cf - mean

        cov = torch.cov(centered.T)

        eps = 1e-5
        cov = cov + eps * torch.eye(cov.size(0))

        sign, logdet = torch.slogdet(cov)

        if sign.item() <= 0:
            cov = cov + 1e-3 * torch.eye(cov.size(0))
            sign, logdet = torch.slogdet(cov)

        inv = torch.inverse(cov)

        means.append(mean)
        inv_covs.append(inv)
        logdets.append(logdet)
        priors.append(torch.tensor(float(Nc) / float(N)))

    means = torch.stack(means, dim=0).to(device)
    inv_covs = torch.stack(inv_covs, dim=0).to(device)
    logdets = torch.stack(logdets, dim=0).to(device)
    priors = torch.stack(priors, dim=0).to(device)

    return means, inv_covs, logdets, priors

@torch.no_grad()
def qda_scores(
    backbone: nn.Module,
    x,
    means,
    inv_covs,
    logdets,
    priors
):
    backbone.eval()

    f = backbone(x)
    scores = []

    for k in range(means.size(0)):
        diff = f - means[k]
        quad = torch.sum((diff @ inv_covs[k]) * diff, dim=1)
        gk = -0.5 * quad - 0.5 * logdets[k] + torch.log(priors[k] + 1e-20)
        scores.append(gk.unsqueeze(1))

    return torch.cat(scores, dim=1)

@torch.no_grad()
def eval_loader_qda_180way(
    model,
    loader,
    m1, inv1, ld1, p1,
    m2, inv2, ld2, p2,
    m3, inv3, ld3, p3,
    m4, inv4, ld4, p4,
    m5, inv5, ld5, p5,
    m6, inv6, ld6, p6,
    m7, inv7, ld7, p7,
    m8, inv8, ld8, p8,
    m9, inv9, ld9, p9
):
    model.eval()

    preds = []
    labels = []

    for x, y in loader:
        x = x.to(device)

        s1 = qda_scores(model.backbone, x, m1, inv1, ld1, p1)
        s2 = qda_scores(model.backbone, x, m2, inv2, ld2, p2)
        s3 = qda_scores(model.backbone, x, m3, inv3, ld3, p3)
        s4 = qda_scores(model.backbone, x, m4, inv4, ld4, p4)
        s5 = qda_scores(model.backbone, x, m5, inv5, ld5, p5)
        s6 = qda_scores(model.backbone, x, m6, inv6, ld6, p6)
        s7 = qda_scores(model.backbone, x, m7, inv7, ld7, p7)
        s8 = qda_scores(model.backbone, x, m8, inv8, ld8, p8)
        s9 = qda_scores(model.backbone, x, m9, inv9, ld9, p9)

        s_all = torch.cat([s1, s2, s3, s4, s5, s6, s7, s8, s9], dim=1)
        pred = torch.argmax(s_all, dim=1)

        preds.extend(pred.detach().cpu().numpy())
        labels.extend(y.detach().cpu().numpy())

    labels_np = np.array(labels)
    preds_np = np.array(preds)

    acc = 100.0 * np.mean(preds_np == labels_np)
    cm = confusion_matrix(
        labels_np,
        preds_np,
        labels=list(range(180))
    )

    return acc, cm

def load_qda_stats_file(path, expected_classes):
    st = torch_load_compat(path, map_location="cpu")

    classes = st.get("classes", expected_classes)

    assert list(classes) == list(expected_classes), (
        f"Expected classes {expected_classes}, got {classes}"
    )

    means = st["means"].to(device)
    inv = st["inv_covs"].to(device)

    if "logdets" in st:
        logdet = st["logdets"].to(device)
    else:
        logdet_list = []

        for k in range(inv.size(0)):
            sign, ld = torch.slogdet(inv[k])
            logdet_list.append(-ld)

        logdet = torch.stack(logdet_list).to(device)

    priors = st.get("priors", None)

    if priors is None:
        priors = torch.ones(inv.size(0), device=device) / float(inv.size(0))
    else:
        priors = priors.to(device)

    return means, inv, logdet, priors

def init_expanded_model_task9():
    state160 = torch_load_compat(
        MD_TASK8_WEIGHTS_PATH,
        map_location=device
    )

    model160 = SingleHeadNet(
        ResNet18Backbone(64),
        160
    ).to(device)

    model160.load_state_dict(state160, strict=True)

    model180 = SingleHeadNet(
        ResNet18Backbone(64),
        180
    ).to(device)

    model180.backbone.load_state_dict(
        model160.backbone.state_dict(),
        strict=True
    )

    with torch.no_grad():
        model180.head.weight[:160, :].copy_(
            model160.head.weight[:160, :]
        )
        model180.head.bias[:160].copy_(
            model160.head.bias[:160]
        )

    return model180

@torch.no_grad()
def normalize_new_classifier_weights_l2(
    model,
    n_old=160,
    eps=1e-12
):
    w = model.head.weight
    w_new = w[n_old:]

    w_flat = w_new.view(w_new.size(0), -1)
    norms = w_flat.norm(2, dim=1, keepdim=True).clamp_min(eps)

    w_flat.div_(norms)

def train_one_config_task9(
    lr_head,
    lr_backbone,
    bs,
    epochs,
    lambda_ewc,
    topk_fisher,
    fisher_neighbors
):
    model = init_expanded_model_task9()

    n_old = 160

    W = model.head.weight
    B = model.head.bias

    weight_mask = torch.ones_like(W, device=W.device)
    bias_mask = torch.ones_like(B, device=B.device)

    weight_mask[:n_old, :] = 0.0
    bias_mask[:n_old] = 0.0

    head_orig = {
        "weight": W.data[:n_old, :].clone(),
        "bias": B.data[:n_old].clone()
    }

    def _freeze_weight_grad(grad):
        return grad * weight_mask.to(grad.device)

    def _freeze_bias_grad(grad):
        return grad * bias_mask.to(grad.device)

    W.register_hook(_freeze_weight_grad)
    B.register_hook(_freeze_bias_grad)

    param_map = {
        n: p
        for n, p in model.named_parameters()
    }

    masks, frozen_idxs, frozen_vals = build_freeze_masks_and_cache(
        model,
        topk_fisher
    )

    first_classes = list(range(0, 20))
    second_classes = list(range(20, 40))
    third_classes = list(range(40, 60))
    fourth_classes = list(range(60, 80))
    fifth_classes = list(range(80, 100))
    sixth_classes = list(range(100, 120))
    seventh_classes = list(range(120, 140))
    eighth_classes = list(range(140, 160))
    ninth_classes = list(range(160, 180))
    all_classes = list(range(180))

    m1, inv1, ld1, p1 = load_qda_stats_file(
        MD_TASK1_STATS_PATH,
        first_classes
    )

    m2, inv2, ld2, p2 = load_qda_stats_file(
        MD_TASK2_STATS_PATH,
        second_classes
    )

    m3, inv3, ld3, p3 = load_qda_stats_file(
        MD_TASK3_STATS_PATH,
        third_classes
    )

    m4, inv4, ld4, p4 = load_qda_stats_file(
        MD_TASK4_STATS_PATH,
        fourth_classes
    )

    m5, inv5, ld5, p5 = load_qda_stats_file(
        MD_TASK5_STATS_PATH,
        fifth_classes
    )

    m6, inv6, ld6, p6 = load_qda_stats_file(
        MD_TASK6_STATS_PATH,
        sixth_classes
    )

    m7, inv7, ld7, p7 = load_qda_stats_file(
        MD_TASK7_STATS_PATH,
        seventh_classes
    )

    m8, inv8, ld8, p8 = load_qda_stats_file(
        MD_TASK8_STATS_PATH,
        eighth_classes
    )

    with torch.no_grad():
        cpu_cache = {
            n: p.view(-1).detach().cpu()
            for n, p in model.named_parameters()
        }

    neighbor_original_values = {}

    for n in fisher_neighbors:
        name = n["name"]
        idx = int(n["index"])

        if name in cpu_cache and idx < cpu_cache[name].numel():
            neighbor_original_values[(name, idx)] = cpu_cache[name][idx]

    ewc_tensors = build_neighbor_tensors(
        model,
        fisher_neighbors,
        neighbor_original_values
    )

    optimizer = optim.SGD([
        {
            "params": model.backbone.parameters(),
            "lr": lr_backbone,
            "weight_decay": WEIGHT_DECAY
        },
        {
            "params": model.head.parameters(),
            "lr": lr_head,
            "weight_decay": 0.0
        }
    ], momentum=MOMENTUM)

    train_set, test_set = get_tinyimg_datasets()

    train_ninth = KeepOriginalLabelsDataset(
        train_set,
        indices_for_classes(train_set, ninth_classes)
    )

    test_first = KeepOriginalLabelsDataset(
        test_set,
        indices_for_classes(test_set, first_classes)
    )

    test_second = KeepOriginalLabelsDataset(
        test_set,
        indices_for_classes(test_set, second_classes)
    )

    test_third = KeepOriginalLabelsDataset(
        test_set,
        indices_for_classes(test_set, third_classes)
    )

    test_fourth = KeepOriginalLabelsDataset(
        test_set,
        indices_for_classes(test_set, fourth_classes)
    )

    test_fifth = KeepOriginalLabelsDataset(
        test_set,
        indices_for_classes(test_set, fifth_classes)
    )

    test_sixth = KeepOriginalLabelsDataset(
        test_set,
        indices_for_classes(test_set, sixth_classes)
    )

    test_seventh = KeepOriginalLabelsDataset(
        test_set,
        indices_for_classes(test_set, seventh_classes)
    )

    test_eighth = KeepOriginalLabelsDataset(
        test_set,
        indices_for_classes(test_set, eighth_classes)
    )

    test_ninth = KeepOriginalLabelsDataset(
        test_set,
        indices_for_classes(test_set, ninth_classes)
    )

    test_all = KeepOriginalLabelsDataset(
        test_set,
        indices_for_classes(test_set, all_classes)
    )

    train_loader = make_loader(train_ninth, bs, True)
    train_ninth_eval_loader = make_loader(train_ninth, 256, False)

    test_first_loader = make_loader(test_first, 256, False)
    test_second_loader = make_loader(test_second, 256, False)
    test_third_loader = make_loader(test_third, 256, False)
    test_fourth_loader = make_loader(test_fourth, 256, False)
    test_fifth_loader = make_loader(test_fifth, 256, False)
    test_sixth_loader = make_loader(test_sixth, 256, False)
    test_seventh_loader = make_loader(test_seventh, 256, False)
    test_eighth_loader = make_loader(test_eighth, 256, False)
    test_ninth_loader = make_loader(test_ninth, 256, False)
    test_all_loader = make_loader(test_all, 256, False)

    best_avg = -1.0
    best_epoch = -1
    best_state = None
    best_results = None

    best_m9 = None
    best_inv9 = None
    best_ld9 = None
    best_p9 = None

    for e in range(1, epochs + 1):
        model.train()

        loss_sum = 0.0
        correct = 0
        total = 0

        for imgs, labels in train_loader:
            imgs = imgs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad(set_to_none=True)

            logits = model(imgs)

            ewc_penalty = 0.0

            if lambda_ewc != 0 and len(ewc_tensors) > 0:
                for name, pack in ewc_tensors.items():
                    p = param_map[name].view(-1)
                    diff = p.index_select(0, pack["idxs"]) - pack["orig"]
                    ewc_penalty += (pack["fish"] * (diff ** 2)).sum()

            loss_ce = F.cross_entropy(logits, labels)
            loss = loss_ce + (lambda_ewc / 2.0) * ewc_penalty

            loss.backward()

            apply_freeze_after_backward(model, masks)

            optimizer.step()

            normalize_new_classifier_weights_l2(
                model,
                n_old=160
            )

            with torch.no_grad():
                W.data[:n_old, :] = head_orig["weight"]
                B.data[:n_old] = head_orig["bias"]

            param_map_after = {
                n: p.data
                for n, p in model.named_parameters()
            }

            apply_strict_freeze_after_step(
                param_map_after,
                frozen_idxs,
                frozen_vals
            )

            preds = logits.argmax(1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)
            loss_sum += float(loss.detach().cpu())

        acc_train_ninth = 100.0 * correct / max(1, total)

        m9, inv9, ld9, p9 = compute_qda_stats_from_loader(
            model.backbone,
            train_ninth_eval_loader,
            class_ids=ninth_classes
        )

        acc_first, cm_first = eval_loader_qda_180way(
            model,
            test_first_loader,
            m1, inv1, ld1, p1,
            m2, inv2, ld2, p2,
            m3, inv3, ld3, p3,
            m4, inv4, ld4, p4,
            m5, inv5, ld5, p5,
            m6, inv6, ld6, p6,
            m7, inv7, ld7, p7,
            m8, inv8, ld8, p8,
            m9, inv9, ld9, p9
        )

        acc_second, cm_second = eval_loader_qda_180way(
            model,
            test_second_loader,
            m1, inv1, ld1, p1,
            m2, inv2, ld2, p2,
            m3, inv3, ld3, p3,
            m4, inv4, ld4, p4,
            m5, inv5, ld5, p5,
            m6, inv6, ld6, p6,
            m7, inv7, ld7, p7,
            m8, inv8, ld8, p8,
            m9, inv9, ld9, p9
        )

        acc_third, cm_third = eval_loader_qda_180way(
            model,
            test_third_loader,
            m1, inv1, ld1, p1,
            m2, inv2, ld2, p2,
            m3, inv3, ld3, p3,
            m4, inv4, ld4, p4,
            m5, inv5, ld5, p5,
            m6, inv6, ld6, p6,
            m7, inv7, ld7, p7,
            m8, inv8, ld8, p8,
            m9, inv9, ld9, p9
        )

        acc_fourth, cm_fourth = eval_loader_qda_180way(
            model,
            test_fourth_loader,
            m1, inv1, ld1, p1,
            m2, inv2, ld2, p2,
            m3, inv3, ld3, p3,
            m4, inv4, ld4, p4,
            m5, inv5, ld5, p5,
            m6, inv6, ld6, p6,
            m7, inv7, ld7, p7,
            m8, inv8, ld8, p8,
            m9, inv9, ld9, p9
        )

        acc_fifth, cm_fifth = eval_loader_qda_180way(
            model,
            test_fifth_loader,
            m1, inv1, ld1, p1,
            m2, inv2, ld2, p2,
            m3, inv3, ld3, p3,
            m4, inv4, ld4, p4,
            m5, inv5, ld5, p5,
            m6, inv6, ld6, p6,
            m7, inv7, ld7, p7,
            m8, inv8, ld8, p8,
            m9, inv9, ld9, p9
        )

        acc_sixth, cm_sixth = eval_loader_qda_180way(
            model,
            test_sixth_loader,
            m1, inv1, ld1, p1,
            m2, inv2, ld2, p2,
            m3, inv3, ld3, p3,
            m4, inv4, ld4, p4,
            m5, inv5, ld5, p5,
            m6, inv6, ld6, p6,
            m7, inv7, ld7, p7,
            m8, inv8, ld8, p8,
            m9, inv9, ld9, p9
        )

        acc_seventh, cm_seventh = eval_loader_qda_180way(
            model,
            test_seventh_loader,
            m1, inv1, ld1, p1,
            m2, inv2, ld2, p2,
            m3, inv3, ld3, p3,
            m4, inv4, ld4, p4,
            m5, inv5, ld5, p5,
            m6, inv6, ld6, p6,
            m7, inv7, ld7, p7,
            m8, inv8, ld8, p8,
            m9, inv9, ld9, p9
        )

        acc_eighth, cm_eighth = eval_loader_qda_180way(
            model,
            test_eighth_loader,
            m1, inv1, ld1, p1,
            m2, inv2, ld2, p2,
            m3, inv3, ld3, p3,
            m4, inv4, ld4, p4,
            m5, inv5, ld5, p5,
            m6, inv6, ld6, p6,
            m7, inv7, ld7, p7,
            m8, inv8, ld8, p8,
            m9, inv9, ld9, p9
        )

        acc_ninth, cm_ninth = eval_loader_qda_180way(
            model,
            test_ninth_loader,
            m1, inv1, ld1, p1,
            m2, inv2, ld2, p2,
            m3, inv3, ld3, p3,
            m4, inv4, ld4, p4,
            m5, inv5, ld5, p5,
            m6, inv6, ld6, p6,
            m7, inv7, ld7, p7,
            m8, inv8, ld8, p8,
            m9, inv9, ld9, p9
        )

        acc_all, cm_all = eval_loader_qda_180way(
            model,
            test_all_loader,
            m1, inv1, ld1, p1,
            m2, inv2, ld2, p2,
            m3, inv3, ld3, p3,
            m4, inv4, ld4, p4,
            m5, inv5, ld5, p5,
            m6, inv6, ld6, p6,
            m7, inv7, ld7, p7,
            m8, inv8, ld8, p8,
            m9, inv9, ld9, p9
        )

        avg_acc = (
            acc_first +
            acc_second +
            acc_third +
            acc_fourth +
            acc_fifth +
            acc_sixth +
            acc_seventh +
            acc_eighth +
            acc_ninth
        ) / 9.0

        print(
            f"Epoch {e:03d} | "
            f"Loss={loss_sum / len(train_loader):.4f} | "
            f"Train(160..179)={acc_train_ninth:.2f}% | "
            f"First(QDA180)={acc_first:.2f}% | "
            f"Second(QDA180)={acc_second:.2f}% | "
            f"Third(QDA180)={acc_third:.2f}% | "
            f"Fourth(QDA180)={acc_fourth:.2f}% | "
            f"Fifth(QDA180)={acc_fifth:.2f}% | "
            f"Sixth(QDA180)={acc_sixth:.2f}% | "
            f"Seventh(QDA180)={acc_seventh:.2f}% | "
            f"Eighth(QDA180)={acc_eighth:.2f}% | "
            f"Ninth(QDA180)={acc_ninth:.2f}% | "
            f"Unified180(QDA)={acc_all:.2f}% | "
            f"Avg={avg_acc:.2f}%"
        )

        if avg_acc > best_avg:
            best_avg = avg_acc
            best_epoch = e

            best_state = {
                k: v.cpu()
                for k, v in model.state_dict().items()
            }

            best_results = {
                "acc_first": acc_first,
                "acc_second": acc_second,
                "acc_third": acc_third,
                "acc_fourth": acc_fourth,
                "acc_fifth": acc_fifth,
                "acc_sixth": acc_sixth,
                "acc_seventh": acc_seventh,
                "acc_eighth": acc_eighth,
                "acc_ninth": acc_ninth,
                "acc_all": acc_all,
                "avg_acc": avg_acc,
                "cm_first": cm_first,
                "cm_second": cm_second,
                "cm_third": cm_third,
                "cm_fourth": cm_fourth,
                "cm_fifth": cm_fifth,
                "cm_sixth": cm_sixth,
                "cm_seventh": cm_seventh,
                "cm_eighth": cm_eighth,
                "cm_ninth": cm_ninth,
                "cm_all": cm_all,
            }

            best_m9 = m9.detach().cpu()
            best_inv9 = inv9.detach().cpu()
            best_ld9 = ld9.detach().cpu()
            best_p9 = p9.detach().cpu()

    print(
        f"[INFO] Best Epoch: {best_epoch} | "
        f"First={best_results['acc_first']:.2f}% | "
        f"Second={best_results['acc_second']:.2f}% | "
        f"Third={best_results['acc_third']:.2f}% | "
        f"Fourth={best_results['acc_fourth']:.2f}% | "
        f"Fifth={best_results['acc_fifth']:.2f}% | "
        f"Sixth={best_results['acc_sixth']:.2f}% | "
        f"Seventh={best_results['acc_seventh']:.2f}% | "
        f"Eighth={best_results['acc_eighth']:.2f}% | "
        f"Ninth={best_results['acc_ninth']:.2f}% | "
        f"Unified180={best_results['acc_all']:.2f}% | "
        f"Avg={best_results['avg_acc']:.2f}%"
    )

    if best_m9 is None:
        raise RuntimeError("No best epoch found; training may have failed.")

    torch.save({
        "means": best_m9,
        "inv_covs": best_inv9,
        "logdets": best_ld9,
        "priors": best_p9,
        "classes": ninth_classes,
        "best_epoch": best_epoch,
        "avg_acc": best_avg,
        "results": {
            "acc_first": best_results["acc_first"],
            "acc_second": best_results["acc_second"],
            "acc_third": best_results["acc_third"],
            "acc_fourth": best_results["acc_fourth"],
            "acc_fifth": best_results["acc_fifth"],
            "acc_sixth": best_results["acc_sixth"],
            "acc_seventh": best_results["acc_seventh"],
            "acc_eighth": best_results["acc_eighth"],
            "acc_ninth": best_results["acc_ninth"],
            "acc_unified180": best_results["acc_all"],
        }
    }, TASK9_BEST_STATS_PATH)

    print(f"[INFO] Saved Task9 QDA stats -> {TASK9_BEST_STATS_PATH}")

    return best_avg, best_state

def grid_search_finetune_task9():
    topk_fisher, fisher_neighbors = load_fisher_data()

    LR_HEADS = [0.5,0.7,0.9]
    LR_BACKBONES = [1e-4,1e-5]
    LAMBDAS = [2]
    BATCH_SIZES = [32]
    EPOCHS_LIST = [2]

    best_acc = -1.0
    best_weights = None
    best_cfg = None

    for lr_h in LR_HEADS:
        for lr_b in LR_BACKBONES:
            for lam in LAMBDAS:
                for bs in BATCH_SIZES:
                    for ep in EPOCHS_LIST:
                        print("\n" + "=" * 70)
                        print(
                            f"🚀 Task9 Config | "
                            f"lr_head={lr_h} | "
                            f"lr_backbone={lr_b} | "
                            f"λ={lam} | "
                            f"bs={bs} | "
                            f"epochs={ep}"
                        )
                        print("=" * 70)

                        acc, weights = train_one_config_task9(
                            lr_head=lr_h,
                            lr_backbone=lr_b,
                            bs=bs,
                            epochs=ep,
                            lambda_ewc=lam,
                            topk_fisher=topk_fisher,
                            fisher_neighbors=fisher_neighbors
                        )

                        if acc > best_acc:
                            best_acc = acc
                            best_weights = deepcopy(weights)
                            best_cfg = {
                                "lr_head": lr_h,
                                "lr_backbone": lr_b,
                                "lambda": lam,
                                "batch_size": bs,
                                "epochs": ep
                            }

    print("\n" + "#" * 70)
    print(f"[DONE] Task9 Best AvgAcc={best_acc:.2f}% | Config={best_cfg}")
    print("#" * 70)

    torch.save(best_weights, MD_TASK9_WEIGHTS_PATH)
    print(f"[INFO] Saved Task9 best weights -> {MD_TASK9_WEIGHTS_PATH}")

if __name__ == "__main__":
    grid_search_finetune_task9()

Mounted at /content/drive
[INFO] Using device: cuda
[INFO] Found processed data at: /content/drive/MyDrive/ML_Project/data/TINYIMG/processed
[INFO] Loaded Fisher data | TopK=1279062 | Neigh=968076

Task9 Config | lr_head=0.5 | lr_backbone=0.0001 | λ=2 | bs=32 | epochs=2
Epoch 001 | Loss=6.9116 | Train(160..179)=13.62% | First(QDA180)=5.50% | Second(QDA180)=0.40% | Third(QDA180)=0.00% | Fourth(QDA180)=1.00% | Fifth(QDA180)=0.40% | Sixth(QDA180)=0.10% | Seventh(QDA180)=0.50% | Eighth(QDA180)=0.30% | Ninth(QDA180)=43.70% | Unified180(QDA)=5.77% | Avg=5.77%
Epoch 002 | Loss=3.1675 | Train(160..179)=22.00% | First(QDA180)=12.00% | Second(QDA180)=1.70% | Third(QDA180)=1.70% | Fourth(QDA180)=5.50% | Fifth(QDA180)=4.30% | Sixth(QDA180)=3.50% | Seventh(QDA180)=6.70% | Eighth(QDA180)=1.60% | Ninth(QDA180)=43.80% | Unified180(QDA)=8.98% | Avg=8.98%
[INFO] Best Epoch: 2 | First=12.00% | Second=1.70% | Third=1.70% | Fourth=5.50% | Fifth=4.30% | Sixth=3.50% | Seventh=6.70% | Eighth=1.60% | Ninth=43.